In [1]:
import math


# ----------------------------
# Variant 21
# I = ∫[1,2] x * (x^2 - 1)^(3/2) dx
# F(x) = (x^2 - 1)^(5/2) / 5
# eps = 1e-4
# ----------------------------

A = 1.0
B = 2.0
EPS = 1e-4

In [2]:
def f(x: float) -> float:
    # x * (x^2 - 1)^(3/2)
    return x * ((x * x - 1.0) ** 1.5)


def F(x: float) -> float:
    # (x^2 - 1)^(5/2) / 5
    return ((x * x - 1.0) ** 2.5) / 5.0


def integral_exact(a: float, b: float) -> float:
    return F(b) - F(a)


# -------- Methods (by n intervals) --------

def left_rect(a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.0
    for i in range(n):
        x = a + i * h
        s += f(x)
    return s * h


def right_rect(a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.0
    for i in range(1, n + 1):
        x = a + i * h
        s += f(x)
    return s * h


def middle_rect(a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.0
    for i in range(n):
        x = a + (i + 0.5) * h
        s += f(x)
    return s * h


def trapezoid(a: float, b: float, n: int) -> float:
    h = (b - a) / n
    s = 0.5 * (f(a) + f(b))
    for i in range(1, n):
        x = a + i * h
        s += f(x)
    return s * h


def simpson(a: float, b: float, n: int) -> float:
    # n must be even
    if n % 2 != 0:
        raise ValueError("Simpson requires even n")
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        x = a + i * h
        s += (4.0 if i % 2 == 1 else 2.0) * f(x)
    return s * h / 3.0

In [3]:
# -------- Double recalculation (step by halving) --------

def double_recalc(a: float, b: float, eps: float, method, n0: int, needs_even: bool = False):
    """
    method: function(a,b,n)->float
    We start with n=n0, then n*=2 until |I_2n - I_n| <= eps.
    Return: (I, n, h)
    """
    n = n0
    if needs_even and (n % 2 != 0):
        n += 1

    I_n = method(a, b, n)
    n2 = n * 2
    if needs_even and (n2 % 2 != 0):
        n2 += 1
    I_2n = method(a, b, n2)

    while abs(I_2n - I_n) > eps:
        n = n2
        I_n = I_2n
        n2 = n * 2
        if needs_even and (n2 % 2 != 0):
            n2 += 1
        I_2n = method(a, b, n2)

    h = (b - a) / n2
    return I_2n, n2, h


def rel_err_percent(I_exact: float, I_approx: float) -> float:
    return abs((I_exact - I_approx) / I_exact) * 100.0


In [4]:
def main():
    I_ex = integral_exact(A, B)

    print("Variant 21")
    print(f"a={A}, b={B}, eps={EPS}")
    print(f"Exact I = {I_ex:.12f}")
    print()

    results = []

    # Start n like typical lab patterns
    methods = [
        ("Метод левых прямоугольников", left_rect, 4, False),
        ("Метод правых прямоугольников", right_rect, 4, False),
        ("Метод центральных прямоугольников", middle_rect, 4, False),
        ("Метод трапеций", trapezoid, 4, False),
        ("Метод Симпсона", simpson, 4, True),
    ]

    for name, m, n0, needs_even in methods:
        I, n, h = double_recalc(A, B, EPS, m, n0, needs_even)
        err = rel_err_percent(I_ex, I)
        results.append((name, I, n, h, err))

    # Print table
    print("Итоговая таблица")
    print("-" * 92)
    print(f"{'Название':35s} | {'Результат':14s} | {'n':8s} | {'h':12s} | {'Погрешн., %':12s}")
    print("-" * 92)
    for name, I, n, h, err in results:
        print(f"{name:35s} | {I:14.8f} | {n:8d} | {h:12.8f} | {err:12.6f}")
    print("-" * 92)

    print()
    print("Проверка точности (|I_2n - I_n| <= eps) выполнена для каждого метода через двойной пересчет.")


if __name__ == "__main__":
    main()

Variant 21
a=1.0, b=2.0, eps=0.0001
Exact I = 3.117691453624

Итоговая таблица
--------------------------------------------------------------------------------------------
Название                            | Результат      | n        | h            | Погрешн., % 
--------------------------------------------------------------------------------------------
Метод левых прямоугольников         |     3.11761217 |    65536 |   0.00001526 |     0.002543
Метод правых прямоугольников        |     3.11777074 |    65536 |   0.00001526 |     0.002543
Метод центральных прямоугольников   |     3.11767498 |      256 |   0.00390625 |     0.000528
Метод трапеций                      |     3.11772442 |      256 |   0.00390625 |     0.001057
Метод Симпсона                      |     3.11769801 |       32 |   0.03125000 |     0.000210
--------------------------------------------------------------------------------------------

Проверка точности (|I_2n - I_n| <= eps) выполнена для каждого метода через дв